In [1]:
!pip install pandas emoji langdetect


[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import json
import os
import re
import unicodedata
from collections import Counter

import pandas as pd
import emoji
from langdetect import detect, LangDetectException


# ============================================================
# cấu hình đường dẫn
# ============================================================
INPUT_JSON_PATH = r"../dataset/quận 5/review_quan_5.json"
OUTPUT_DIR = r"../dataset/preprocessed_q5"

# OUTPUT_REVIEWS_CSV = "review_quan_5_preprocessed.csv"
# OUTPUT_EMOJI_ANALYSIS_CSV = "review_quan_5_emoji_analysis.csv"


OUTPUT_REVIEWS_JSON = "review_quan_5_preprocessed.json"
OUTPUT_EMOJI_ANALYSIS_JSON = "review_quan_5_emoji_analysis.json"

MANUAL_DICT_PATH = r"../dataset/preprocessed/tu_can_xu_ly_thu_cong_da_dich.csv"

from urllib.parse import urlparse, parse_qsl, urlencode, urlunparse
import hashlib

def canonicalize_google_maps_url(url):
    if pd.isna(url) or str(url).strip() == "":
        return ""

    url = str(url).strip()

    try:
        parsed = urlparse(url)

        drop_params = {"hl", "authuser", "rclk"}

        query_pairs = parse_qsl(parsed.query, keep_blank_values=True)
        filtered_pairs = [(k, v) for k, v in query_pairs if k not in drop_params]
        new_query = urlencode(filtered_pairs)

        canonical = urlunparse((
            parsed.scheme.lower(),
            parsed.netloc.lower(),
            parsed.path,
            parsed.params,
            new_query,
            ""
        ))

        return canonical
    except Exception:
        return url

def make_restaurant_id(url):
    canonical_url = canonicalize_google_maps_url(url)
    if canonical_url == "":
        return None
    return hashlib.md5(canonical_url.encode("utf-8")).hexdigest()
# ============================================================
# hàm hỗ trợ
# ============================================================
def convert_time_to_days(time_str):
    if pd.isna(time_str):
        return None

    s = str(time_str).lower().strip()
    s = s.replace("thời gian chỉnh sửa:", "").strip()
    s = s.replace("một", "1").replace("vài", "1")

    nums = re.findall(r"\d+", s)
    if not nums:
        return 1

    num = int(nums[0])

    if "năm" in s:
        return num * 365
    elif "tháng" in s:
        return num * 30
    elif "tuần" in s:
        return num * 7
    elif "ngày" in s:
        return num
    elif "giờ" in s or "phút" in s or "giây" in s:
        return 1

    return None


fixed_encoding = []
fixed_invisible = []
fixed_whitespace = []
fixed_non_alpha = []


def track_and_clean(text):
    if pd.isna(text) or not isinstance(text, str):
        return ""

    step_0 = text

    # chuẩn hóa unicode
    step_0 = unicodedata.normalize("NFC", step_0)

    # sửa lỗi mã hóa
    step_1 = step_0
    try:
        decoded = step_0.encode("latin1").decode("utf-8")
        if decoded != step_0:
            step_1 = decoded
            fixed_encoding.append({"old": step_0, "new": step_1})
    except Exception:
        pass

    # loại bỏ ký tự ẩn
    step_2 = re.sub(r"[\u200b\u200c\u200d\u200e\u200f\ufeff]+", "", step_1)
    step_2 = re.sub(r"[\x00-\x08\x0b\x0c\x0e-\x1f\x7f]", "", step_2)

    if step_2 != step_1:
        fixed_invisible.append({"old": repr(step_1), "new": repr(step_2)})

    # xử lý khoảng trắng
    step_3 = re.sub(r"[\r\n\t]+", " ", step_2)
    step_3 = re.sub(r"\s+", " ", step_3).strip()

    if step_3 != step_2:
        fixed_whitespace.append({"old": repr(step_2), "new": repr(step_3)})

    # chuyển về chữ thường
    step_4 = step_3.lower()

    # xử lý dấu câu, giữ dấu phẩy
    step_5 = step_4.replace(".", ",")
    step_5 = re.sub(r"[^\w\s,]|_|\d", " ", step_5)
    step_5 = re.sub(r"\s+", " ", step_5).strip()

    if step_5 != step_4:
        fixed_non_alpha.append({"old": step_4, "new": step_5})

    return step_5


def extract_emojis(text):
    if not isinstance(text, str):
        return []
    return [e["emoji"] for e in emoji.emoji_list(text)]


emotion_mapping = {
    "❤": " yêu_thích ", "❤️": " yêu_thích ", "💚": " yêu_thích ", "🤍": " yêu_thích ",
    "💕": " yêu_thích ", "💖": " yêu_thích ", "💗": " yêu_thích ", "💓": " yêu_thích ",
    "🩷": " yêu_thích ", "💛": " yêu_thích ", "🩵": " yêu_thích ", "🤎": " yêu_thích ",
    "♥️": " yêu_thích ", "❣️": " yêu_thích ", "💞": " yêu_thích ", "💘": " yêu_thích ",
    "🫶": " thả_tim ", "🫶🏻": " thả_tim ", "🫶🏼": " thả_tim ", "🫶🏿": " thả_tim ",

    "😍": " tuyệt_vời ", "🥰": " đáng_yêu ", "🤩": " xuất_sắc ",
    "👍": " tốt ", "👍🏻": " tốt ", "👍🏼": " tốt ",
    "👌": " đồng_ý ", "👌🏻": " đồng_ý ", "👌🏼": " đồng_ý ",
    "👏": " tán_thưởng ", "👏🏻": " tán_thưởng ", "👏🏼": " tán_thưởng ",
    "💯": " hoàn_hảo ", "⭐": " sao_đánh_giá ", "🌟": " sao_tuyệt_vời ",
    "✨": " lấp_lánh ", "🔥": " cực_đỉnh ", "✅": " tốt ", "✔️": " tốt ",

    "😊": " vui_vẻ ", "😁": " vui_vẻ ", "😃": " thích_thú ", "😄": " vui_vẻ ",
    "😀": " vui_vẻ ", "🙂": " hài_lòng ", "☺️": " hài_lòng ", "😆": " cười_lớn ",
    "😂": " buồn_cười ", "🤣": " buồn_cười ", "🤭": " cười_tủm_tỉm ",
    "😅": " cười_ngại ", "😉": " nháy_mắt ", "😜": " vui_nhộn ", "🤪": " vui_nhộn ",
    "😘": " hôn_gió ", "😚": " hôn_gió ", "😙": " hôn_gió ", "😗": " hôn_gió ",
    "😌": " nhẹ_nhõm ", "😇": " ngoan_ngoãn ",

    "😋": " ngon_miệng ", "🤤": " thèm_thuồng ",

    "🥲": " cảm_động ", "🥹": " rưng_rưng ", "🥺": " mong_chờ ",

    "👎": " tệ ", "👎🏻": " tệ ",
    "😡": " tức_giận ", "🤬": " chửi_rủa ", "😤": " bực_bội ", "😠": " bực_mình ",
    "😑": " chán_nản ", "😒": " không_hài_lòng ", "🙄": " chán_ngán ", "😐": " bình_thường ",
    "❌": " không_tốt ", "⚠️": " cảnh_báo ",

    "😭": " khóc_lớn ", "😢": " buồn_khóc ", "😞": " thất_vọng ", "😔": " chán_nản ",
    "😓": " mệt_mỏi ", "😥": " buồn_bực ", "😫": " mệt_mỏi ", "😣": " khó_chịu ",

    "🤢": " buồn_nôn ", "🤮": " kinh_tởm ", "🤯": " bất_ngờ ", "😱": " hoảng_hốt ",
    "😳": " bối_rối ", "😨": " sợ_hãi ",
}


def filter_and_convert_emojis(text):
    if pd.isna(text) or not isinstance(text, str):
        return ""

    for emj, text_replacement in emotion_mapping.items():
        if emj in text:
            text = text.replace(emj, text_replacement)

    text = emoji.replace_emoji(text, replace="")
    text = re.sub(r"\s+", " ", text).strip()
    return text


def contains_emoji(text):
    if pd.isna(text) or not isinstance(text, str):
        return False
    return len(emoji.emoji_list(text)) > 0


def remove_elongation(text):
    if pd.isna(text) or not isinstance(text, str):
        return ""

    pattern = r"([a-zđáàãảạâấầẫẩậăắằẵẳặéèẽẻẹêếềễểệíìĩỉịóòõỏọôốồỗổộơớờỡởợúùũủụưứừữửựýỳỹỷỵ])\1{2,}"
    text = re.sub(pattern, r"\1", text, flags=re.IGNORECASE)

    text = re.sub(r",+", ",", text)
    text = re.sub(r"\s*,\s*", ", ", text)
    text = re.sub(r"\s+", " ", text).strip()

    return text


ABBREV_DICT = {
    "ko": "không", "k": "không", "kg": "không", "kog": "không",
    "khg": "không", "hong": "không", "hok": "không", "khom": "không", "khong": "không",
    "vs": "với", "cx": "cũng", "cg": "cũng", "cũg": "cũng", "cux": "cũng",
    "trc": "trước", "truoc": "trước", "th": "thì",
    "lun": "luôn", "luon": "luôn", "luonn": "luôn", "luônn": "luôn",
    "r": "rồi", "rùi": "rồi", "rui": "rồi", "ròi": "rồi", "xog": "xong",
    "z": "vậy", "vz": "vậy", "zậy": "vậy", "zui": "vui", "dzui": "vui",
    "j": "gì", "qá": "quá", "qa": "quá", "quáa": "quá", "rât": "rất",
    "nhma": "nhưng mà", "nma": "nhưng mà", "nhg": "nhưng",
    "nchung": "nói chung", "nc": "nói chung", "nch": "nói chung",
    "bt": "bình thường", "bth": "bình thường", "bthg": "bình thường", "bthuong": "bình thường",
    "vc": "vô cùng", "vcl": "vô cùng", "vl": "vô cùng", "vch": "vô cùng",
    "ntn": "như thế nào", "ms": "mới",

    "nv": "nhân viên", "nvien": "nhân viên", "nvpv": "nhân viên phục vụ",
    "ng": "người", "ngta": "người ta", "ngừoi": "người", "nguoi": "người", "ngươ": "người",
    "mk": "mình", "mik": "mình", "tui": "tôi", "t": "tôi",
    "mn": "mọi người", "mng": "mọi người", "mnguoi": "mọi người",
    "bn": "bạn", "b": "bạn", "ae": "anh em", "ce": "chị em",
    "gd": "gia đình", "gđ": "gia đình", "gdinh": "gia đình",

    "đc": "được", "dc": "được", "dk": "được", "đk": "được", "đươc": "được",
    "ql": "quản lý", "qly": "quản lý", "cskh": "chăm sóc khách hàng",
    "pv": "phục vụ", "pvu": "phục vụ", "phuc": "phục vụ",
    "kh": "khách hàng",
    "ch": "cửa hàng", "nh": "nhà hàng", "cn": "chi nhánh",
    "đt": "điện thoại", "dt": "điện thoại",
    "ck": "chuyển khoản", "tt": "thanh toán", "vat": "thuế VAT",
    "km": "khuyến mãi", "sp": "sản phẩm", "cty": "công ty",
    "đb": "đặc biệt", "dbiet": "đặc biệt",
    "kgian": "không gian", "csvc": "cơ sở vật chất",
    "dth": "dễ thương", "dthw": "dễ thương", "dthuong": "dễ thương",
    "mlem": "ngon", "thik": "thích", "thix": "thích",
    "hssv": "học sinh sinh viên", "hs": "học sinh", "sv": "sinh viên",
    "sn": "sinh nhật",
    "thit": "thịt", "nuoc": "nước", "nhiet": "nhiệt", "thuong": "thương",

    "hnay": "hôm nay", "hm": "hôm nay",
    "hqua": "hôm qua", "hqa": "hôm qua",
    "tgian": "thời gian", "tg": "thời gian",
    "hcm": "Hồ Chí Minh", "tphcm": "TP. Hồ Chí Minh",
    "hn": "Hà Nội", "sg": "Sài Gòn", "tp": "thành phố",
    "vn": "Việt Nam", "hq": "Hàn Quốc",
    "pmh": "Phú Mỹ Hưng", "svh": "Sư Vạn Hạnh",

    "ok": "ok", "oke": "ok", "okela": "ok", "okila": "ok", "okey": "ok", "oki": "ok",
    "ngonn": "ngon", "lắmm": "lắm",
    "nhaa": "nha", "ạa": "ạ", "ạaa": "ạ", "nhee": "nhé",
    "tr": "trời", "chời": "trời",
    "haha": "haha", "hihi": "hihi", "huhu": "huhu",

    "rcm": "recommend", "rec": "recommend", "recomend": "recommend", "reccomend": "recommend",
    "ord": "order", "oder": "order",
    "rv": "review",
    "cmt": "comment", "rep": "reply",
    "fb": "facebook", "ig": "instagram", "ib": "inbox",
    "wf": "wifi", "cf": "cafe",
    "stt": "status", "app": "application",
    "toping": "topping", "toppings": "topping",
    "dishes": "dish", "noodles": "noodle", "prices": "price", "services": "service",
    "takecare": "take care", "chilli": "chili",

    "panchan": "banchan", "pancha": "banchan",
    "kimbab": "kimbap",
    "tokbokki": "tokbokki", "tok": "tokbokki", "tteokbokki": "tokbokki", "tobokki": "tokbokki",
    "phomai": "phô mai",
    "vnd": "VNĐ", "vnđ": "VNĐ",
    "nvs": "WC", "wc": "WC",
    "qr": "QR",
}


def normalize_text(text: str) -> str:
    if not isinstance(text, str):
        return ""

    words = text.split()
    normalized = []
    for w in words:
        w_lower = w.lower()
        if w_lower in ABBREV_DICT:
            normalized.append(ABBREV_DICT[w_lower])
        else:
            normalized.append(w)

    return " ".join(normalized)


def load_manual_dict(dict_path):
    try:
        df_dict = pd.read_csv(dict_path)
        print(f"Đã đọc file từ điển thủ công với {len(df_dict)} dòng.")

        df_dict_valid = df_dict.dropna(subset=["Dịch"])
        abbreviation_mapping = dict(
            zip(
                df_dict_valid["N-gram"].astype(str).str.lower().str.strip(),
                df_dict_valid["Dịch"].astype(str).str.lower().str.strip(),
            )
        )
        print(f"Đã tạo manual dictionary với {len(abbreviation_mapping)} cặp từ.")
        return abbreviation_mapping
    except Exception as e:
        print(f"Không đọc được file từ điển thủ công: {e}")
        return {}


def replace_words_from_dict(text, abbreviation_mapping):
    if pd.isna(text) or not isinstance(text, str):
        return ""

    words = text.split()
    replaced_words = [abbreviation_mapping.get(word.lower(), word) for word in words]
    return " ".join(replaced_words)


def filter_language(text):
    if pd.isna(text) or text.strip() == "":
        return "unknown"
    try:
        return detect(text)
    except LangDetectException:
        return "unknown"


# ============================================================
# bước 1: đọc json và bung thành bảng review
# ============================================================
# with open(INPUT_JSON_PATH, "r", encoding="utf-8") as f:
#     data = json.load(f)

# reviews_data = []

# for restaurant in data:
#     restaurant_name = restaurant.get("name")
#     reviews = restaurant.get("reviews", [])

#     for review in reviews:
#         features = review.get("features", {})

#         reviews_data.append({
#             "restaurant_name": restaurant_name,
#             "reviewer": review.get("reviewer"),
#             "rating": review.get("rating"),
#             "time": review.get("time"),
#             "comment": review.get("comment"),
#             "food_rating": features.get("Đồ ăn"),
#             "service_rating": features.get("Dịch vụ"),
#             "atmosphere_rating": features.get("Bầu không khí"),
#         })

with open(INPUT_JSON_PATH, "r", encoding="utf-8") as f:
    data = json.load(f)

reviews_data = []

for restaurant in data:
    restaurant_name = restaurant.get("name")
    restaurant_url = restaurant.get("url", "")
    restaurant_url_source = restaurant.get("url_source", "")

    source_url_for_id = restaurant_url if str(restaurant_url).strip() else restaurant_url_source
    canonical_url = canonicalize_google_maps_url(source_url_for_id)
    restaurant_id = make_restaurant_id(source_url_for_id)

    reviews = restaurant.get("reviews", [])

    for review in reviews:
        features = review.get("features", {})

        reviews_data.append({
            "restaurant_id": restaurant_id,
            "restaurant_url": source_url_for_id,
            "canonical_url": canonical_url,
            "restaurant_name": restaurant_name,
            "reviewer": review.get("reviewer"),
            "rating": review.get("rating"),
            "time": review.get("time"),
            "comment": review.get("comment"),
            "food_rating": features.get("Đồ ăn"),
            "service_rating": features.get("Dịch vụ"),
            "atmosphere_rating": features.get("Bầu không khí"),
        })

df_reviews = pd.DataFrame(reviews_data)

print("Đã load xong dữ liệu review.")
print(f"Kích thước ban đầu: {df_reviews.shape}")
print(df_reviews.head())



META_JSON_PATH = r"../dataset/metadata_restaurant.json"

df_meta = pd.read_json(META_JSON_PATH)

valid_restaurant_ids = set(df_meta["restaurant_id"].dropna().unique())

before_count = len(df_reviews)
df_reviews = df_reviews[df_reviews["restaurant_id"].isin(valid_restaurant_ids)].reset_index(drop=True)
after_count = len(df_reviews)

print("\n--- LỌC REVIEW THEO METADATA ĐÃ GIỮ LẠI ---")
print("Số restaurant_id hợp lệ từ metadata:", len(valid_restaurant_ids))
print("Số review trước khi lọc:", before_count)
print("Số review sau khi lọc:", after_count)
print("Số review bị loại:", before_count - after_count)

# ============================================================
# bước 2: xóa trùng lặp
# ============================================================
print(f"\nKích thước trước khi xóa trùng: {df_reviews.shape[0]} dòng")
is_duplicated_rev = df_reviews.duplicated(
    subset=["restaurant_name", "reviewer", "comment"],
    keep="first"
)
num_duplicates_rev = is_duplicated_rev.sum()

if num_duplicates_rev > 0:
    print(f"Phát hiện {num_duplicates_rev} review trùng lặp. Đang xóa...")
    df_reviews = df_reviews.drop_duplicates(
        subset=["restaurant_name", "reviewer", "comment"],
        keep="first"
    ).reset_index(drop=True)
else:
    print("Không có review trùng lặp.")

print(f"Kích thước sau khi xóa trùng: {df_reviews.shape[0]} dòng")


# ============================================================
# bước 3: chuyển time -> days_ago
# ============================================================
df_reviews["time"] = df_reviews["time"].apply(convert_time_to_days).astype("Int64")
df_reviews.rename(columns={"time": "days_ago"}, inplace=True)

print("\nĐã chuyển cột time thành days_ago.")
print(df_reviews[["restaurant_name", "days_ago", "comment"]].head())


# ============================================================
# bước 4: tạo cột comment_clean, giữ nguyên comment gốc
# ============================================================
df_reviews["comment"] = df_reviews["comment"].fillna("").astype(str)
df_reviews["comment_clean"] = df_reviews["comment"]

print("\nĐã tạo cột comment_clean từ comment gốc.")


# ============================================================
# bước 5: lọc comment quá dài
# vẫn làm như cũ, nhưng tính trên comment_clean
# ============================================================
# df_reviews["word_count"] = df_reviews["comment_clean"].apply(lambda x: len(str(x).split()))

# initial_count = len(df_reviews)
# df_reviews = df_reviews[df_reviews["word_count"] <= 256].reset_index(drop=True)

# removed_count = initial_count - len(df_reviews)
# max_len_after = df_reviews["word_count"].max() if len(df_reviews) > 0 else 0

# print("\n--- THỐNG KÊ LỌC ĐỘ DÀI ---")
# print(f"Đã xóa: {removed_count} review quá dài.")
# print(f"Còn lại: {len(df_reviews)} review.")
# print(f"Review dài nhất hiện tại: {max_len_after} từ.")


# ============================================================
# bước 6: chuẩn hóa unicode, encoding, ký tự ẩn, dấu câu...
# chạy trên comment_clean
# ============================================================
print(f"\nĐang dọn dẹp {len(df_reviews)} comment_clean...")
initial_count = len(df_reviews)

df_reviews["comment_clean"] = df_reviews["comment_clean"].apply(track_and_clean)

df_reviews = df_reviews[df_reviews["comment_clean"] != ""].reset_index(drop=True)
empty_removed_count = initial_count - len(df_reviews)

print("\n--- THỐNG KÊ TRACK AND CLEAN ---")
print(f"Lỗi encoding sửa được: {len(fixed_encoding)}")
print(f"Ký tự ẩn xử lý được: {len(fixed_invisible)}")
print(f"Lỗi khoảng trắng xử lý được: {len(fixed_whitespace)}")
print(f"Dấu câu / chữ số xử lý được: {len(fixed_non_alpha)}")
print(f"Review rỗng bị xóa: {empty_removed_count}")

print("\nVí dụ comment gốc và comment_clean sau bước này:")
print(df_reviews[["comment", "comment_clean"]].head())


# ============================================================
# bước 7: phân tích emoji trên comment_clean
# ============================================================
print(f"\nBắt đầu phân tích emoji trên {len(df_reviews):,} dòng review")

df_reviews["emojis_found"] = df_reviews["comment_clean"].apply(extract_emojis)
df_reviews["emoji_count"] = df_reviews["emojis_found"].apply(len)
df_reviews["unique_emoji_in_review"] = df_reviews["emojis_found"].apply(lambda x: len(set(x)))
df_reviews["has_emoji"] = df_reviews["emoji_count"] > 0

total_rows = len(df_reviews)
reviews_with_emoji = df_reviews["has_emoji"].sum()
all_emojis = [e for lst in df_reviews["emojis_found"] for e in lst]
total_emoji_occurrences = len(all_emojis)
unique_emoji_types = len(set(all_emojis))

summary = pd.DataFrame({
    "Chỉ số": [
        "Tổng số dòng review",
        "Review có chứa emoji",
        "Tỷ lệ review có emoji (%)",
        "Tổng số emoji xuất hiện",
        "Số loại emoji khác nhau"
    ],
    "Giá trị": [
        total_rows,
        int(reviews_with_emoji),
        round(reviews_with_emoji / total_rows * 100, 2) if total_rows else 0,
        total_emoji_occurrences,
        unique_emoji_types
    ]
})

print("\n--- TÓM TẮT EMOJI ---")
print(summary)

emoji_counter = Counter(all_emojis)
emoji_df = pd.DataFrame([
    {
        "emoji": e,
        "ten_emoji": emoji.demojize(e, language="en").strip(":").replace("_", " "),
        "so_lan_xuat_hien": cnt,
        "phan_tram": round(cnt / total_emoji_occurrences * 100, 2) if total_emoji_occurrences else 0,
        "so_review_chua": sum(1 for lst in df_reviews["emojis_found"] if e in lst),
    }
    for e, cnt in emoji_counter.most_common()
])

print(f"Tổng số loại emoji: {len(emoji_df)}")
print(emoji_df.head(10))


# ============================================================
# bước 8: thay emoji cảm xúc, xóa emoji còn lại
# chạy trên comment_clean
# ============================================================
print("\nĐang giữ lại emoji cảm xúc và xóa emoji nhiễu...")

df_reviews["comment_clean_after_emoji"] = df_reviews["comment_clean"].apply(filter_and_convert_emojis)

print("\nVí dụ review có emoji trước/sau:")
print(
    df_reviews[df_reviews["has_emoji"]][["comment_clean", "comment_clean_after_emoji"]].head(10)
)

df_reviews["comment_clean"] = df_reviews["comment_clean_after_emoji"]
df_reviews.drop(columns=["comment_clean_after_emoji"], inplace=True)

print("\nĐã xử lý emoji xong.")


# ============================================================
# bước 9: kiểm tra còn emoji sót lại không
# ============================================================
remaining_emojis_df = df_reviews[df_reviews["comment_clean"].apply(contains_emoji)]

if len(remaining_emojis_df) == 0:
    print("\nKhông còn emoji sót lại trong comment_clean.")
else:
    print(f"\nCẢNH BÁO: Vẫn còn {len(remaining_emojis_df)} review còn emoji.")
    print(remaining_emojis_df[["comment_clean"]].head(10))


# ============================================================
# bước 10: xóa ký tự kéo dài, chuẩn hóa dấu phẩy
# chạy trên comment_clean
# ============================================================
df_reviews["comment_clean_old"] = df_reviews["comment_clean"]

print("\nĐang xử lý ký tự kéo dài và dấu phẩy liên tiếp...")
df_reviews["comment_clean"] = df_reviews["comment_clean"].apply(remove_elongation)

changed_rows = (df_reviews["comment_clean"] != df_reviews["comment_clean_old"]).sum()

print(f"Tổng số comment_clean bị thay đổi: {changed_rows}")

if changed_rows > 0:
    changed_df = df_reviews[df_reviews["comment_clean"] != df_reviews["comment_clean_old"]]
    print("\nVí dụ trước/sau:")
    for idx, (before, after) in enumerate(
        zip(changed_df["comment_clean_old"].head(5), changed_df["comment_clean"].head(5)), 1
    ):
        print(f"\n{idx}. Trước: {before[:150]}")
        print(f"   Sau:   {after[:150]}")

df_reviews.drop(columns=["comment_clean_old"], inplace=True)


# ============================================================
# bước 11: normalize viết tắt bằng dictionary cứng
# chạy trên comment_clean
# ============================================================
df_reviews = df_reviews.dropna(subset=["comment_clean"]).copy()
df_reviews["comment_clean"] = df_reviews["comment_clean"].apply(normalize_text)

print("\nĐã hoàn thành normalize viết tắt bằng ABBREV_DICT.")


# ============================================================
# bước 12: thay thế tiếp bằng file từ điển thủ công
# chạy trên comment_clean
# ============================================================
abbreviation_mapping = load_manual_dict(MANUAL_DICT_PATH)

if abbreviation_mapping:
    print("\nĐang thay thế theo file từ điển thủ công...")
    df_reviews["comment_clean"] = df_reviews["comment_clean"].apply(
        lambda x: replace_words_from_dict(x, abbreviation_mapping)
    )
    print("Đã thay thế xong bằng file từ điển thủ công.")
else:
    print("\nBỏ qua bước manual dictionary vì không đọc được file.")


# ============================================================
# bước 13: lọc ngôn ngữ
# chạy trên comment_clean
# ============================================================
print(f"\nBắt đầu lọc ngôn ngữ. Tổng số review hiện tại: {len(df_reviews)}")

df_reviews["language"] = df_reviews["comment_clean"].apply(filter_language)

df_valid_lang = df_reviews[df_reviews["language"].isin(["vi", "en"])].copy()
removed_lang_count = len(df_reviews) - len(df_valid_lang)
df_reviews = df_valid_lang.drop(columns=["language"]).reset_index(drop=True)

print(f"Đã loại bỏ {removed_lang_count} review không phải tiếng Việt / tiếng Anh.")
print(f"Tổng số review còn lại: {len(df_reviews)}")


# ============================================================
# bước 14: cập nhật lại word_count sau toàn bộ tiền xử lý
# tính trên comment_clean
# ============================================================
# df_reviews["word_count"] = df_reviews["comment_clean"].apply(lambda x: len(str(x).split()))

# initial_count = len(df_reviews)
# df_reviews = df_reviews[df_reviews["word_count"] <= 256].reset_index(drop=True)

# removed_count = initial_count - len(df_reviews)
# max_len_after = df_reviews["word_count"].max() if len(df_reviews) > 0 else 0

# print("\n--- THỐNG KÊ SAU TIỀN XỬ LÝ ---")
# print(f"Đã xóa thêm: {removed_count} review quá dài sau chuẩn hóa.")
# print(f"Còn lại: {len(df_reviews)} review.")
# print(f"Review dài nhất hiện tại: {max_len_after} từ.")


# ============================================================
# bước 15: dọn các cột tạm và lưu file
# không có bước gán label
# ============================================================
cols_to_drop = [
    "emojis_found",
    "emoji_count",
    "unique_emoji_in_review",
    "has_emoji"
]
existing_cols_to_drop = [col for col in cols_to_drop if col in df_reviews.columns]
df_reviews.drop(columns=existing_cols_to_drop, inplace=True)

if not os.path.exists(OUTPUT_DIR):
    os.makedirs(OUTPUT_DIR)

# output_reviews_path = os.path.join(OUTPUT_DIR, OUTPUT_REVIEWS_CSV)
# output_emoji_path = os.path.join(OUTPUT_DIR, OUTPUT_EMOJI_ANALYSIS_CSV)

# df_reviews.to_csv(output_reviews_path, index=False, encoding="utf-8-sig")
# emoji_df.to_csv(output_emoji_path, index=False, encoding="utf-8-sig")

output_reviews_path = os.path.join(OUTPUT_DIR, OUTPUT_REVIEWS_JSON)
output_emoji_path = os.path.join(OUTPUT_DIR, OUTPUT_EMOJI_ANALYSIS_JSON)

df_reviews.to_json(
    output_reviews_path,
    orient="records",
    force_ascii=False,
    indent=2
)

emoji_df.to_json(
    output_emoji_path,
    orient="records",
    force_ascii=False,
    indent=2
)

print("\nĐã lưu file thành công.")
print(f"File review đã tiền xử lý: {output_reviews_path}")
print(f"File phân tích emoji: {output_emoji_path}")

print("\nCác cột cuối cùng trong df_reviews:")
print(df_reviews.columns.tolist())

print("\n5 dòng đầu sau cùng:")
print(df_reviews.head())

Đã load xong dữ liệu review.
Kích thước ban đầu: (13553, 11)
                      restaurant_id  \
0  e7b09f8c49e9533d4d79737ebd508a96   
1  e7b09f8c49e9533d4d79737ebd508a96   
2  e7b09f8c49e9533d4d79737ebd508a96   
3  e7b09f8c49e9533d4d79737ebd508a96   
4  e7b09f8c49e9533d4d79737ebd508a96   

                                      restaurant_url  \
0  https://www.google.com/maps/place/Qu%C3%A1n+%C...   
1  https://www.google.com/maps/place/Qu%C3%A1n+%C...   
2  https://www.google.com/maps/place/Qu%C3%A1n+%C...   
3  https://www.google.com/maps/place/Qu%C3%A1n+%C...   
4  https://www.google.com/maps/place/Qu%C3%A1n+%C...   

                                       canonical_url  restaurant_name  \
0  https://www.google.com/maps/place/Qu%C3%A1n+%C...  Quán ǎn Hùng Ký   
1  https://www.google.com/maps/place/Qu%C3%A1n+%C...  Quán ǎn Hùng Ký   
2  https://www.google.com/maps/place/Qu%C3%A1n+%C...  Quán ǎn Hùng Ký   
3  https://www.google.com/maps/place/Qu%C3%A1n+%C...  Quán ǎn Hùng Ký   
4 

In [3]:
check_df = df_reviews[["restaurant_name", "restaurant_url", "canonical_url", "restaurant_id"]].drop_duplicates()

print("Số quán duy nhất theo name/url:", len(check_df))
print("Số restaurant_id duy nhất:", check_df["restaurant_id"].nunique())

collision_name = check_df.groupby("restaurant_id")["restaurant_name"].nunique()
bad_name = collision_name[collision_name > 1]

print("\nSố restaurant_id gắn với hơn 1 tên quán:", len(bad_name))
if len(bad_name) > 0:
    print(bad_name)
    display(
        check_df[check_df["restaurant_id"].isin(bad_name.index)]
        .sort_values(["restaurant_id", "restaurant_name"])
    )
else:
    print("Không có collision theo tên quán.")

collision_url = check_df.groupby("restaurant_id")["canonical_url"].nunique()
bad_url = collision_url[collision_url > 1]

print("\nSố restaurant_id gắn với hơn 1 canonical_url:", len(bad_url))
if len(bad_url) > 0:
    print(bad_url)
    display(
        check_df[check_df["restaurant_id"].isin(bad_url.index)]
        .sort_values(["restaurant_id", "canonical_url"])
    )
else:
    print("Không có collision theo canonical_url.")

Số quán duy nhất theo name/url: 341
Số restaurant_id duy nhất: 341

Số restaurant_id gắn với hơn 1 tên quán: 0
Không có collision theo tên quán.

Số restaurant_id gắn với hơn 1 canonical_url: 0
Không có collision theo canonical_url.


In [4]:
print("Số quán duy nhất trong review:", df_reviews["restaurant_id"].nunique())

Số quán duy nhất trong review: 341


In [5]:
import pandas as pd

META_JSON_PATH = r"../dataset/metadata_restaurant.json"

df_meta = pd.read_json(META_JSON_PATH)

print("Số dòng metadata:", len(df_meta))
print(df_meta.columns.tolist())
df_meta.head()
meta_ids = set(df_meta["restaurant_id"].dropna().unique())
review_ids = set(df_reviews["restaurant_id"].dropna().unique())

print("Số quán trong metadata:", len(meta_ids))
print("Số quán trong review:", len(review_ids))
print("Số quán có trong metadata nhưng không có review:", len(meta_ids - review_ids))
print("Số quán có trong review nhưng không có trong metadata:", len(review_ids - meta_ids))

Số dòng metadata: 349
['url', 'name', 'avg_rating', 'review_count', 'category', 'price_range', 'address', 'phone', 'opening_hours', 'lgbtq_friendly', 'popular_times', 'url_source', 'about', 'category_clean', 'source_url_for_id', 'canonical_url', 'restaurant_id', 'price_min', 'price_max', 'price_level', 'rating', 'opening_parsed', 'popular_parsed', 'about_clean']
Số quán trong metadata: 349
Số quán trong review: 341
Số quán có trong metadata nhưng không có review: 8
Số quán có trong review nhưng không có trong metadata: 0
